# probad · GPT reference run (letter log-probs)

Scores the own-format experiments with an OpenAI chat model, **the same way as the small models**: the prompt is sent as-is and we read the log-probability of each answer letter (`A`, `B`, `C`) for the first token of the reply (`top_logprobs=20`).

- Run the cells **one at a time**. Each experiment cell prints its estimated cost before it starts and its real cost after.
- The order follows the priority list: **x1, x4** (knowing vs using) → **x5, x2** (option picks, negation) → **x3, x7**.
- Results go to `results/<model>/raw/<label>.csv` in the **same format as the run notebook**, so the analysis cells and the cross-model plots (`COMPARE_DIRS`) work with them.
- Every API answer is cached in `cache.jsonl`: re-running a cell after a crash costs nothing for prompts already answered.

In [ ]:
# ================================================================ Configuration (edit this cell)
import os, sys, re, json, math, time, hashlib, inspect
from pathlib import Path

MODEL = "gpt-6-luna"            # must return logprobs: gpt-6-luna with reasoning off, or gpt-4.1 / gpt-4.1-mini / gpt-4o-mini
REASONING_EFFORT = "none"       # "none" turns reasoning off (needed for logprobs on GPT-6). Set None for gpt-4.1 / gpt-4o models
PRICE_IN, PRICE_CACHED, PRICE_OUT = 0.10, 0.01, 0.50    # $ per 1M tokens for MODEL (standard tier); check the pricing page

INSTRUCT = None                 # None = exact same prompt as the small models. If letter mass is low in the pilot,
                                # set e.g. "Reply with only the letter of your answer." (added as a system message)
ORDERS = "swap"                 # "swap" = seed 0 + swapped orders (gold in every position); "seed0" = half the cost
TOP_LOGPROBS = 20               # maximum the API allows
MAX_WORKERS = 8                 # parallel requests; lower it if you hit rate limits
N_ITEMS = None                  # None = all 600 items; e.g. 20 for a quick try

ON_KAGGLE = Path("/kaggle").exists()
DATA_PATH = Path(os.environ.get("PROBAD_DATA", "/kaggle/input/datasets/amallick0507/probad"))
RESULTS_ROOT = Path(os.environ.get("PROBAD_RESULTS", "/kaggle/working/results" if ON_KAGGLE else "results"))
RUN_NAME = MODEL + ("" if N_ITEMS is None else f"_smoke{N_ITEMS}")
OUT = RESULTS_ROOT / RUN_NAME
RAW = OUT / "raw"
RAW.mkdir(parents=True, exist_ok=True)
print("results ->", OUT)

## Setup
On Kaggle: clones the repo with the `GITHUB_TOKEN` secret and reads `OPENAI_API_KEY` from the secrets. Elsewhere: set `PROBAD_REPO` to a local checkout and `OPENAI_API_KEY` in the environment.

In [ ]:
import subprocess
REPO_USER, REPO_NAME = "AnikMallick", "bn-proverbs-in-context"
REPO_DIR = (Path("/kaggle/working") / REPO_NAME) if ON_KAGGLE else Path(os.environ.get("PROBAD_REPO", ".")).resolve()

if ON_KAGGLE:
    import base64
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    if not REPO_DIR.exists():
        token = secrets.get_secret("GITHUB_TOKEN")
        auth = base64.b64encode(f"{REPO_USER}:{token}".encode()).decode()
        try:
            subprocess.run(["git", "-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {auth}",
                            "clone", f"https://github.com/{REPO_USER}/{REPO_NAME}.git", str(REPO_DIR)], check=True)
        finally:
            del token, auth
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR), "openai"], check=True)
    os.environ["OPENAI_API_KEY"] = secrets.get_secret("OPENAI_API_KEY")

for p in (REPO_DIR / "src", REPO_DIR):
    if (p / "probad").exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from openai import OpenAI

import probad.builder as exp
from probad.schema import TestItem

client = OpenAI()
pd.set_option("display.width", 200)

## Data

In [ ]:
ITEMS = [it.model_dump() for it in TestItem.load_items(DATA_PATH / "test.jsonl")]
ITEMS = ITEMS[:N_ITEMS] if N_ITEMS else ITEMS
print(len(ITEMS), "items,", len({it["proverb"]["proverb_id"] for it in ITEMS}), "proverbs")

EXPERIMENTS = {"x1": exp.x1_correct_vs_literal, "x2": exp.x2_negative, "x3": exp.x3_none_of_these,
               "x4": exp.x4_meaning_no_context, "x5": exp.x5_three_way, "x7": exp.x7_no_question}

## Helpers
- `build`: one prompt from the repo's builder (accepts the builder's 2-, 3- or 4-tuple).
- `order_seeds`: seed 0 plus the first seeds that move the gold letter to every other position (x3: gold is always C, so it swaps the other two).
- `ask`: one API call (cached), returns the top-20 first tokens with log-probs.
- `letter_scores`: the probability of each shown letter. Tokens like `A`, ` A`, `A.` all count as A. A letter missing from the top 20 gets the lowest top-20 log-prob (an upper bound) and is counted in `n_missing`.

In [ ]:
KEYS = ("correct", "literal", "plausible", "none")
_norm = lambda s: re.sub(r"\s+", " ", str(s)).strip().strip("।.!?\"'“”‘’ ").lower()

def build(fn, item, lang, seed):
    """-> (system, prompt, gold) or None. Builders without a seed argument have one order only."""
    ps = inspect.signature(fn).parameters
    if "seed" in ps or any(p.kind == p.VAR_KEYWORD for p in ps.values()):
        out = fn(item, lang=lang, seed=seed)
    elif seed == 0:
        out = fn(item, lang=lang)
    else:
        return None
    if out is None:
        return None
    out = tuple(out)
    if len(out) == 2:
        return None, out[0], out[1]
    if len(out) == 3:            # (prompt, gold, options) or (system, prompt, gold)
        return (None, out[0], out[1]) if isinstance(out[2], (list, tuple)) else out
    return out[0], out[1], out[2]

def option_lines(prompt):
    """{letter: option text} of the last question ("A. ..." lines)."""
    block = prompt[prompt.rfind("\n\n"):] if "\n\n" in prompt else prompt
    return {m.group(1): m.group(2).strip() for m in re.finditer(r"^([ABC])\. (.*)$", block, flags=re.M)}

def option_key(text, item):
    """Which option a text is (correct / literal / plausible); anything else is 'none' (None of these)."""
    cands = {"correct": [item["proverb"].get("meaning", {}).get(l) for l in ("en", "bn")], "literal": [], "plausible": []}
    for l in ("en", "bn"):
        o = item.get(l, {}).get("options", {})
        for k in ("correct", "literal", "plausible"):
            cands[k].append(o.get(k))
    t = _norm(text)
    for k, texts in cands.items():
        if any(c and _norm(c) == t for c in texts):
            return k
    for k, texts in cands.items():
        for c in texts:
            c = _norm(c) if c else ""
            if c and min(len(c), len(t)) >= 12 and (c in t or t in c):
                return k
    return "none"

def order_seeds(fn, item, lang):
    first = build(fn, item, lang, 0)
    if first is None:
        return []
    seeds = [0]
    if ORDERS != "swap":
        return seeds
    n = len(option_lines(first[1]))
    golds, prompts = {str(first[2]).upper()}, {first[1]}
    for s in range(1, 300):
        b = build(fn, item, lang, s)
        if b and str(b[2]).upper() not in golds:
            seeds.append(s); golds.add(str(b[2]).upper()); prompts.add(b[1])
        if len(golds) >= n:
            break
    if len(seeds) == 1:          # gold never moves (x3): take the first seed with a different option order
        for s in range(1, 300):
            b = build(fn, item, lang, s)
            if b and b[1] not in prompts:
                seeds.append(s)
                break
    return seeds

# ---- API (with a cache on disk, so nothing is paid for twice)
CACHE_FILE = OUT / "cache.jsonl"
CACHE = {}
if CACHE_FILE.exists():
    for line in CACHE_FILE.read_text(encoding="utf-8").splitlines():
        r = json.loads(line); CACHE[r["key"]] = r
print(len(CACHE), "cached answers")

def messages(system, prompt):
    sys_text = "\n\n".join(x for x in (system, INSTRUCT) if x)
    return ([{"role": "system", "content": sys_text}] if sys_text else []) + [{"role": "user", "content": prompt}]

def request_key(msgs):
    return hashlib.md5(json.dumps([MODEL, msgs], ensure_ascii=False).encode()).hexdigest()

def ask(msgs, retries=4):
    key = request_key(msgs)
    if key in CACHE:
        return CACHE[key]
    for attempt in range(retries):
        try:
            extra = {"reasoning_effort": REASONING_EFFORT} if REASONING_EFFORT else {}
            r = client.chat.completions.create(model=MODEL, messages=msgs, temperature=0, seed=0,
                                               max_completion_tokens=1, logprobs=True, top_logprobs=TOP_LOGPROBS, **extra)
            break
        except Exception as e:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt * 3)
    if r.choices[0].logprobs is None:
        raise RuntimeError(f"{MODEL} returned no logprobs; use a model that supports them (e.g. gpt-4.1-mini)")
    u = r.usage
    cached = (u.prompt_tokens_details.cached_tokens or 0) if u.prompt_tokens_details else 0
    rec = {"key": key, "model": r.model,
           "top": [{"token": t.token, "logprob": t.logprob} for t in r.choices[0].logprobs.content[0].top_logprobs],
           "in": u.prompt_tokens, "cached": cached, "out": u.completion_tokens}
    return rec

def cost(recs):
    return sum((r["in"] - r["cached"]) * PRICE_IN + r["cached"] * PRICE_CACHED + r["out"] * PRICE_OUT for r in recs) / 1e6

# ---- turn one answer into a result row (same columns as the run notebook)
def clean(tok):
    return tok.strip().strip("*").rstrip(".):").upper()

def letter_scores(top, letters):
    found = {}
    for t in top:
        L = clean(t["token"])
        if L in letters:
            found[L] = found.get(L, 0.0) + math.exp(t["logprob"])
    floor = min(t["logprob"] for t in top)
    raw = {L: math.log(found[L]) if L in found else floor for L in letters}
    return raw, sum(found.values()), len(letters) - len(found)

def score_row(item, prompt, gold, rec):
    lines = option_lines(prompt); letters = list(lines)
    keys = {L: option_key(t, item) for L, t in lines.items()}
    raw, mass, n_missing = letter_scores(rec["top"], letters)
    tot = sum(math.exp(v) for v in raw.values())
    g = str(gold).strip().upper()
    pred = max(raw, key=raw.get)
    row = {"gold": g.lower(), "pred": pred.lower(), "correct": pred == g, "n_opts": len(letters),
           "margin": raw[g] - max(v for L, v in raw.items() if L != g), "letter_mass": mass,
           "top_token": rec["top"][0]["token"], "top_is_letter": clean(rec["top"][0]["token"]) in letters,
           "n_missing": n_missing, "gold_key": keys[g], "pred_key": keys[pred]}
    for L in "ABC":
        row[f"p_{L.lower()}"] = math.exp(raw[L]) / tot if L in raw else np.nan
        row[f"key_{L.lower()}"] = keys.get(L)
    for k in KEYS:
        row[f"p_{k}"] = sum(row[f"p_{L.lower()}"] for L in letters if keys[L] == k) if k in keys.values() else np.nan
    return row

def meta(e, lang, it, seed, order):
    m = it.get("meta", {}) or {}
    return {"model": MODEL, "run": RUN_NAME, "exp": e, "label": f"{e}_{lang}", "item_id": it["item_id"],
            "proverb_id": it["proverb"]["proverb_id"], "figurative": it["proverb"].get("figurative"),
            "split": m.get("split"), "subsets": "|".join(m.get("subsets") or []), "conv_lang": lang,
            "render": None, "prompt_lang": None, "condition": None, "seed": seed, "order": order}

## The run function
`run("x1", "bn")`: builds all prompts, prints the number of requests and an estimated cost (from the average tokens seen so far), asks the API, saves `raw/x1_bn.csv` and prints accuracy, letter mass and the real cost. If the CSV already exists it is loaded instead (use `overwrite=True` to redo it; cached answers are still free).

In [ ]:
SPENT = []      # cost of every condition run in this session

def run(e, lang, overwrite=False, limit=None):
    label = f"{e}_{lang}"
    path = RAW / f"{label}.csv"
    if path.exists() and not overwrite:
        df = pd.read_csv(path); print(f"{label}: already done ({len(df)} rows), loaded from {path.name}")
        return df
    items = ITEMS[:limit] if limit else ITEMS
    jobs = []
    for it in items:
        for order, s in enumerate(order_seeds(EXPERIMENTS[e], it, lang)):
            system, prompt, gold = build(EXPERIMENTS[e], it, lang, s)
            jobs.append((it, s, order, system, prompt, gold, messages(system, prompt)))
    todo = {request_key(j[6]): j[6] for j in jobs if request_key(j[6]) not in CACHE}
    seen = list(CACHE.values())
    avg_in = np.mean([r["in"] for r in seen]) if seen else float("nan")
    print(f"{label}: {len(jobs)} scorings, {len(todo)} new requests "
          f"(~{avg_in:.0f} input tokens each -> ~${len(todo) * (avg_in * PRICE_IN + PRICE_OUT) / 1e6:.3f})"
          if seen else f"{label}: {len(jobs)} scorings, {len(todo)} new requests (cost estimate after the first run)")

    new, errors = [], 0
    with open(CACHE_FILE, "a", encoding="utf-8") as f, ThreadPoolExecutor(MAX_WORKERS) as pool:
        futures = {pool.submit(ask, m): k for k, m in todo.items()}
        for fut in tqdm(as_completed(futures), total=len(futures), desc=label):
            try:
                rec = fut.result()
                CACHE[rec["key"]] = rec; new.append(rec)
                f.write(json.dumps(rec, ensure_ascii=False) + "\n"); f.flush()
            except Exception as ex:
                errors += 1; print("error:", str(ex)[:200])

    rows = [{**meta(e, lang, it, s, order), **score_row(it, prompt, gold, CACHE[request_key(m)])}
            for it, s, order, system, prompt, gold, m in jobs if request_key(m) in CACHE]
    df = pd.DataFrame(rows)
    if errors:
        print(f"{errors} requests failed: CSV NOT saved. Re-run the cell (answers so far are cached).")
    elif not limit:
        df.to_csv(path, index=False)
    c = cost(new); SPENT.append((label, c))
    item_acc = df.groupby("item_id")["correct"].mean()
    print(f"{label}: acc seed0 {df[df.order == 0].correct.mean():.3f} | acc (mean over orders) {item_acc.mean():.3f} | "
          f"same result in every order {(item_acc.isin([0, 1])).mean():.0%} | letter mass median "
          f"{df.letter_mass.median():.3f} | letters missing from top-20 {df.n_missing.mean():.2f} | "
          f"cost ${c:.3f} (session total ${sum(x for _, x in SPENT):.3f})")
    return df

## 0. Pilot: 20 items of x1_bn, first order only
Check before spending more:
- **letter mass** (summed probability of the letters) near 1 and `top_token` a letter: the model answers with a letter by itself, so `INSTRUCT = None` is fine.
- letter mass low and `top_token` like "The", "**", "উত্তর": set `INSTRUCT` in the config cell, re-run the config cell and the cells below it.
- the cost line: multiply by ~(600 / 20) × number of orders to get the cost of one full condition.

In [ ]:
_orders = ORDERS
ORDERS = "seed0"
pilot = run("x1", "bn", limit=20)
ORDERS = _orders
display(pilot[["item_id", "gold", "pred", "correct", "letter_mass", "n_missing", "top_token", "p_a", "p_b"]])

## Priority 1: knowing vs using (x1, x4)

In [ ]:
x1_en = run("x1", "en")

In [ ]:
x1_bn = run("x1", "bn")

In [ ]:
x4_en = run("x4", "en")

In [ ]:
x4_bn = run("x4", "bn")

## Priority 2: three options and negation (x5, x2)

In [ ]:
x5_en = run("x5", "en")

In [ ]:
x5_bn = run("x5", "bn")

In [ ]:
x2_en = run("x2", "en")

In [ ]:
x2_bn = run("x2", "bn")

## Priority 3: none-of-these and no question (x3, x7)

In [ ]:
x3_en = run("x3", "en")

In [ ]:
x3_bn = run("x3", "bn")

In [ ]:
x7_en = run("x7", "en")

In [ ]:
x7_bn = run("x7", "bn")

## Summary
Reads every CSV in `raw/`. Cost is from the cache, so it covers every answer paid for with this model, across sessions.

In [ ]:
dfs = [pd.read_csv(p) for p in sorted(RAW.glob("*.csv"))]
if dfs:
    allr = pd.concat(dfs, ignore_index=True)
    item = allr.groupby(["label", "item_id"]).agg(correct=("correct", "mean"), mass=("letter_mass", "mean"),
                                                   missing=("n_missing", "mean")).reset_index()
    summary = item.groupby("label").agg(items=("item_id", "size"), acc=("correct", "mean"),
                                        acc_strict=("correct", lambda s: (s == 1).mean()),
                                        consistent=("correct", lambda s: s.isin([0, 1]).mean()),
                                        letter_mass=("mass", "median"), missing=("missing", "mean"))
    summary.insert(1, "acc_seed0", allr[allr.order == 0].groupby("label")["correct"].mean())
    display(summary.round(3))
recs = list(CACHE.values())
print(f"{len(recs)} answers paid for with {MODEL}: {sum(r['in'] for r in recs)/1e6:.2f}M input tokens, "
      f"total cost ${cost(recs):.2f}")
print("For the cross-model plots: add", OUT, "to COMPARE_DIRS in the run notebook.")